# Exploratory Data Analysis — Pharmaceutical Cold Chain

Run from the project root. This notebook reads the supplied CSV without modifying it and focuses on data quality, class balance, feature distributions, and target relationships. Interpret associations descriptively, not causally.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

ROOT = Path.cwd()
if not (ROOT / 'data').exists(): ROOT = ROOT / 'Pharmaceutical_Cold_Chain_Project'
DATA = ROOT / 'data' / 'Pharmaceutical_Cold_Chain_Dataset.csv'
df = pd.read_csv(DATA, encoding_errors='replace')
TARGET = 'Excursion_Before_Delivery'
sns.set_theme(style='whitegrid')
print('Shape:', df.shape)
print('Columns:', df.columns.tolist())
display(df.head(), df.tail())

## Dataset overview and data types

In [ ]:
display(df.dtypes.rename('dtype').to_frame())
display(df.describe(include='all').T)
numeric = df.select_dtypes(include='number').columns.tolist()
categorical = df.select_dtypes(exclude='number').columns.tolist()
print('Numeric:', numeric)
print('Categorical:', categorical)
print('Target valid:', TARGET in df and set(df[TARGET].dropna().unique()).issubset({0,1}))

## Data quality assessment

In [ ]:
display(df.isna().sum().sort_values(ascending=False).rename('missing').to_frame())
print('Exact duplicate rows:', df.duplicated().sum())
for c in categorical:
    print(f'\n{c}: {df[c].nunique()} unique')
    display(df[c].value_counts(dropna=False).to_frame('count'))
print('Target values:', df[TARGET].value_counts(dropna=False).to_dict())
print('Binary target only:', set(df[TARGET].dropna().unique()).issubset({0,1}))

## Univariate distributions and outlier screening
Boxplots are a screening aid; extremes are not automatically invalid without domain limits.

In [ ]:
for c in numeric:
    if c == TARGET: continue
    fig, ax = plt.subplots(1, 2, figsize=(10, 3))
    sns.histplot(df[c], kde=True, ax=ax[0]); ax[0].set_title(f'{c} distribution')
    sns.boxplot(x=df[c], ax=ax[1]); ax[1].set_title('Outlier screening')
    plt.tight_layout(); plt.show()

## Target distribution and class imbalance

In [ ]:
target_counts = df[TARGET].value_counts().sort_index()
display(pd.DataFrame({'count': target_counts, 'percent': 100*target_counts/len(df)}))
sns.countplot(data=df, x=TARGET); plt.title('Excursion before delivery'); plt.show()

## Feature relationships with target
The listed fields were present in the inspected CSV. Temperature summaries/trend, exposure risk, and threshold proximity may be outcome-adjacent; visualize them for EDA, but exclude them from predictors until feature timing is verified.

In [ ]:
numeric_candidates = [c for c in ['Ambient_Temperature_C','Coolant_Quantity_g','Delivery_Distance_km','Thermal_Holding_Capacity_Min','Temperature_Rate_of_Change','Temperature_Threshold_Proximity','Avg_Temperature_C','Max_Temperature_C','Temperature_Std_Dev'] if c in df]
for c in numeric_candidates:
    sns.boxplot(data=df, x=TARGET, y=c); plt.title(f'{c} by excursion outcome'); plt.show()
categorical_candidates = [c for c in ['Insulation_Level','Temperature_Trend','Sensor_Anomaly_Flag','Refrigeration_Status','Courier_Type','Weather_Condition','Temperature_Exposure_Risk'] if c in df]
for c in categorical_candidates:
    rate = df.groupby(c, dropna=False)[TARGET].mean().sort_values(ascending=False)
    ax=rate.plot(kind='bar', figsize=(8,3)); ax.set_ylabel('Excursion fraction'); ax.set_title(f'Excursion fraction by {c}'); plt.tight_layout(); plt.show()

## Correlation heatmap and findings
Correlations are descriptive and do not establish causation. Strong associations involving observed temperature summaries may reflect label construction.

In [ ]:
corr = df.select_dtypes(include='number').corr()
plt.figure(figsize=(14,10)); sns.heatmap(corr, cmap='coolwarm', center=0); plt.title('Numeric feature correlation'); plt.tight_layout(); plt.show()
print('Potentially constant columns:', [c for c in df if df[c].nunique(dropna=False)==1])
print('Target prevalence:', round(df[TARGET].mean()*100, 2), '%')
print('Review reports/EDA_Report.md for data-timing and leakage notes.')